---
title: "Chapter 5: Performance"
short_title: "5. Performance"
---

## 1. Measurement method

- **Environment:** Google Colab, **CPU-only** runtime (no GPU), with the same code for every ID range.
- **Execution:** `chunked_crawl.py` crawls the ID ranges of the members **one after another** on a single runtime, one crawler at a time (arXiv's rate limits apply to all machines as a whole). Each range is split into steps: discovery, chunks of 1,000 papers (metadata, source download, references), reference submission dates from Kaggle, and finally a **sweep** of the whole range for the work the chunks left behind (files arXiv temporarily refused, failed API batches). Once every range is done, each range with remaining work receives one **late sweep** per rerun of the command. Every step runs under `benchmark.py` and is saved to Drive when it ends, so when Colab disconnects (12-hour limit) a new runtime restores the data and resumes from the unfinished step.
- **Time:** the wall time of each invocation is measured with `time.perf_counter()` from start-up to crawler exit, including the waits imposed by rate limits and the retry passes. Each step (late sweeps included) is one invocation; the time of a range is the **sum of its completed invocations** (`benchmark_runs.py`), and the time of the group is the sum over all ranges. This sum excludes snapshot saving and restoring, environment setup, idle time between Colab sessions, and any step interrupted by a runtime failure (such a step is rerun from the beginning in the next session). [Section 6](#outside-wall-time) measures these parts separately.
- **RAM:** the RSS of the **whole crawler process tree** (main process and every child process), sampled every 2 s with `psutil`. Peak RAM is the largest sample and average RAM is the mean of the samples.
- **Disk:** the size of the data folder, scanned every 300 s (equivalent to `du`). Peak disk is the largest scan, and the output size is the last scan when the crawler exits.
- **Whole group:** time, papers and output size are summed, while peak RAM is the maximum over the ranges (only one crawler runs at a time). **Peak disk** is the largest total size of the folders on disk at any moment, since the folders of completed ranges remain on the runtime's disk (`group_benchmark.py --one-machine`).

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, one_machine, pct, provenance, show, table)
provenance()

In [ ]:
m, g = members(), group()
stages = load("run_log")
stages["seconds"] = pd.to_numeric(stages["seconds"], errors="coerce")
discovery = stages[stages["stage"] == "discovery"].groupby("member")["seconds"].sum()

def row(r, disc):
    return {"Member": r["member"], "Papers": f"{r['papers']:,}", "Invocations": r["invocations"],
            "Total wall time": human_seconds(r["wall_seconds"]),
            "Mean time per paper": human_seconds(r["seconds_per_paper"]),
            "Discovery time": human_seconds(disc),
            "Peak RAM": human_bytes(r["ram_peak_bytes"]), "Average RAM": human_bytes(r["ram_mean_bytes"]),
            "Peak disk": human_bytes(r["data_dir_peak_bytes"]),
            "Output size": human_bytes(r["data_dir_final_bytes"])}

rows = [row(r, discovery.get(r["member"])) for _, r in m.iterrows()]
rows.append(row(g, discovery.sum()))
rows[-1]["Member"] = "Group"
table(pd.DataFrame(rows), "Table 5.1. Running time and memory footprint per member")
clock = "wall_seconds" if m["wall_seconds"].notna().all() else "crawler_seconds"
longest = m[clock].max()
if one_machine():
    print(f"The ranges ran one after another on a single runtime: crawling the data of the whole group took "
          f"{human_seconds(g[clock])} (longest range: {human_seconds(longest)}).")
else:
    print(f"The members ran in parallel on separate runtimes: the group's data was complete after "
          f"{human_seconds(longest)} (longest member); the total machine time of the group is "
          f"{human_seconds(g[clock])}.")
if clock == "crawler_seconds":
    print("Note: some members have no benchmark.py measurements, so times are compared on the main.py clock.")

"Mean time per paper" is the wall time divided by the number of existing papers. It is the **throughput** of the whole pipeline, including the waits imposed by the rate limits of arXiv and Semantic Scholar, not the latency of processing a single paper.

## 2. Time per stage

Stage times are recorded in `run_log.csv` and summed over all invocations. The source download and reference stages run in parallel, so the sum of the stage times exceeds the wall time.

In [ ]:
by_stage = stages.groupby(["member", "stage"])["seconds"].sum().reset_index()
by_stage["hours"] = by_stage["seconds"] / 3600
fig = px.bar(by_stage, x="member", y="hours", color="stage", barmode="group", color_discrete_sequence=COLORS,
             title="Total time per stage and member",
             labels={"member": "Member", "hours": "Hours", "stage": "Stage"})
show(fig)

## 3. Source processing time per paper

Total time to download and extract the versions of one paper (`total_seconds` in `versions.json`), excluding the time spent queued behind the rate limits:

In [ ]:
versions = load("versions")
per_paper = (versions[versions["status"].isin(FINAL_SOURCE)]
             .groupby(["member", "arxiv_id"])["total_seconds"].sum().reset_index())
fig = px.box(per_paper, x="member", y="total_seconds", log_y=True, color_discrete_sequence=COLORS,
             title="Source download and extraction time per paper",
             labels={"member": "Member", "total_seconds": "Seconds (log scale)"})
show(fig)
d = per_paper["total_seconds"]
table(pd.DataFrame({"Statistic": ["Papers", "Mean", "Median", "95th percentile", "Maximum"],
                    "Value": [f"{len(d):,}", human_seconds(d.mean()), human_seconds(d.median()),
                              human_seconds(d.quantile(0.95)), human_seconds(d.max())]}),
      "Table 5.2. Source processing time per paper")

## 3b. End-to-end processing time of a single paper (standalone runs)

The mean time per paper in Table 5.1 is the throughput of the bulk run. To obtain the time needed to process **one** paper from start to finish (discovery, metadata, download and extraction of every version, references), a random sample of papers was rerun one paper at a time in fresh folders (`per_paper_benchmark.py`), separately from the main run.

In [ ]:
if meta().get("paper_timings"):
    pt = load("paper_timings")
    pt["wall_seconds"] = pd.to_numeric(pt["wall_seconds"], errors="coerce")
    ok = pt[pt["status"] == "complete"]["wall_seconds"]
    table(pd.DataFrame({"Statistic": ["Papers in the sample", "Papers completed", "Mean", "Median", "Minimum", "Maximum"],
                        "Value": [f"{len(pt):,}", f"{len(ok):,}", human_seconds(ok.mean()), human_seconds(ok.median()),
                                  human_seconds(ok.min()), human_seconds(ok.max())]}),
          "Table 5.2b. End-to-end processing time of a single paper, standalone runs")
else:
    print("No data yet: run step 8b of the crawler notebook and export data/m1 again.")

## 4. RAM and disk usage over time

Each line is one `benchmark.py` invocation (samples were thinned at export).

In [ ]:
samples = load("performance_samples")
for col in ["elapsed_seconds", "rss_tree_bytes", "data_dir_bytes"]:
    samples[col] = pd.to_numeric(samples[col], errors="coerce")
samples["hours"] = samples["elapsed_seconds"] / 3600
samples["RAM (MB)"] = samples["rss_tree_bytes"].map(mib)
samples["invocation"] = samples["member"] + " · " + samples["run_id"].astype(str)
fig = px.line(samples, x="hours", y="RAM (MB)", color="invocation", color_discrete_sequence=COLORS,
              title="RAM of the crawler process tree over time",
              labels={"hours": "Time since the start of the invocation (hours)"})
show(fig)
disk = samples.dropna(subset=["data_dir_bytes"]).copy()
disk["Data folder (GB)"] = disk["data_dir_bytes"] / 2**30
fig = px.line(disk, x="hours", y="Data folder (GB)", color="invocation", markers=True,
              color_discrete_sequence=COLORS, title="Size of the data folder over time",
              labels={"hours": "Time since the start of the invocation (hours)"})
show(fig)

## 5. Invocations and arXiv rate limits

In [ ]:
runs = load("benchmark_runs")
# the step of each run: the snapshot saved right after the run ended
snaps = load("snapshot_log")
snaps["seconds"] = pd.to_numeric(snaps["seconds"], errors="coerce")
saves = snaps[snaps["action"] == "save"].assign(saved=lambda d: pd.to_datetime(d["time"], utc=True))
runs["ended"] = pd.to_datetime(runs["ended_utc"], utc=True)
runs = (pd.merge_asof(runs.sort_values("ended"), saves[["member", "saved", "step"]].sort_values("saved"),
                      left_on="ended", right_on="saved", by="member", direction="forward",
                      tolerance=pd.Timedelta("15min"))
        .sort_values("started_utc").reset_index(drop=True))
runs_view = pd.DataFrame({
    "Member": runs["member"], "Step": runs["step"].fillna("(unknown)"), "run_id": runs["run_id"],
    "Exit code": runs["return_code"],
    "Interrupted": runs["interrupted"], "Wall time": runs["wall_seconds"].map(human_seconds),
    "Peak RAM": runs["ram_sampled_peak_bytes"].map(human_bytes),
    "Peak disk": runs["data_dir_sampled_peak_bytes"].map(human_bytes)})
table(runs_view, "Table 5.3. Every benchmark.py invocation")
req = pd.DataFrame({
    "Member": m["member"], "arXiv API requests": m["arxiv_api_requests"],
    "Source download requests": m["arxiv_download_requests"],
    "Minimum time at arXiv's pace": m["arxiv_floor_seconds"].map(human_seconds),
    "Time paused by arXiv refusals": m["arxiv_pause_seconds"].map(human_seconds),
    "Global pauses": m["arxiv_pauses"],
    "Route pauses": m["arxiv_route_pauses"]})
table(req, "Table 5.4. Requests sent to arXiv")

"Minimum time at arXiv's pace" is the time the requests actually sent would need at the pace required by arXiv (3 s per API call and 0.5 s per source request on average). "Time paused by arXiv refusals" covers only the pauses of every request; while a single route pauses, the other routes keep working, so no time is lost ([Chapter 2](./02_methodology.ipynb), Section 7). Most of the wall time is spent waiting for the rate limits and for pauses after arXiv refusals rather than on computation, which is why the RAM of the crawler stays low and nearly constant throughout the run.

(outside-wall-time)=
## 6. Time outside the wall time, and late sweeps

Three parts are not included in the total wall time of Table 5.1 and are measured separately here: saving a snapshot to Drive after each step and restoring the data folders on a new runtime (`snapshot_log.csv`), and a step interrupted when the runtime died (warning of `group_benchmark.py`). Environment setup and idle time between Colab sessions are not measured.

In [ ]:
import re
dead = []
for line in meta()["warnings"]:
    hit = re.match(r"(\w+): invocation (\S+) has no final summary.*last seen after ([\d.]+) s", line)
    if hit:
        dead.append((hit.group(1), float(hit.group(3))))
dead = pd.DataFrame(dead, columns=["member", "seconds"])

def count(n, word, seconds):
    return f"{n} {word}{'' if n == 1 else 's'}, {human_seconds(seconds)}" if n else "0"

def outside(name):
    s = snaps[snaps["member"] == name]
    d = dead[dead["member"] == name]
    saved, restored = s["action"] == "save", s["action"] == "restore"
    return {"Member": name,
            "Snapshot saves": count(int(saved.sum()), "save", s.loc[saved, "seconds"].sum()),
            "Restores": count(int(restored.sum()), "restore", s.loc[restored, "seconds"].sum()),
            "Interrupted steps": count(len(d), "step", d["seconds"].sum())}

table(pd.DataFrame([outside(n) for n in m["member"]]), "Table 5.5. Time outside the total wall time")
extra = snaps["seconds"].sum() + dead["seconds"].sum()
print(f"Total wall time of the group (Table 5.1):     {human_seconds(g[clock])}")
print(f"+ snapshot saves and restores:               {human_seconds(snaps['seconds'].sum())}")
print(f"+ steps interrupted by a runtime failure:     {human_seconds(dead['seconds'].sum())}")
print(f"= including the time outside the wall time:   {human_seconds(g[clock] + extra)}")

late = runs[runs["step"].fillna("").str.startswith("late sweep")]
if len(late):
    lv = late.groupby("member")["wall_seconds"].agg(["count", "sum"]).reset_index()
    table(pd.DataFrame({"Member": lv["member"], "Late sweeps": lv["count"],
                        "Total wall time": lv["sum"].map(human_seconds)}),
          "Table 5.6. Late sweeps (already included in the wall time of Table 5.1)")
    with_failed = set(versions.loc[versions["status"] == "failed", "member"])
    stuck = [n for n in lv["member"] if n in with_failed]
    done = [n for n in lv["member"] if n not in with_failed]
    if stuck:
        print(f"{', '.join(stuck)}: the late sweeps only retried the failed versions, i.e. the HTML submissions "
              "(Chapter 6, Section 3);")
        print("arXiv returned the same file every time, so the outcome did not change.")
    if done:
        print(f"{', '.join(done)}: the late sweep completed the work left by the sweep; no work remained afterwards.")